# Week 3 — Data Preprocessing

This notebook handles missing values, encodes categorical fields, normalizes numeric features, and creates a time-ordered train/test split. The most recent month is always the test set; `TRAINING_MONTHS` controls how many immediately preceding months are used for training.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path('Data and Python File for Internship')
OUTPUT_DIR = Path('outputs')
OUTPUT_FILE = OUTPUT_DIR / '02_preprocessed.csv'
DATA_PATTERN = 'CRMLSSold*.csv'
TRAINING_MONTHS = 3
CATEGORICAL_COLUMNS = ['PropertyType', 'PropertySubType']
NUMERIC_COLUMNS = ['ClosePrice', 'LivingArea', 'Bedrooms', 'Bathrooms', 'LotSize']
FEATURE_COLUMNS = ['LivingArea', 'Bedrooms', 'Bathrooms', 'LotSize', 'PropertyType', 'PropertySubType']
REQUIRED_COLUMNS = ['PropertyType', 'PropertySubType', 'CloseDate', 'ClosePrice', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet']

In [2]:
csv_files = sorted(DATA_DIR.glob(DATA_PATTERN))
if not csv_files:
    raise FileNotFoundError(f'No sold CSV files found under {DATA_DIR.resolve()}.')

frames = [pd.read_csv(csv_file, usecols=REQUIRED_COLUMNS, low_memory=False) for csv_file in csv_files]
raw_df = pd.concat(frames, ignore_index=True, sort=False)
df = raw_df.rename(columns={
    'BedroomsTotal': 'Bedrooms',
    'BathroomsTotalInteger': 'Bathrooms',
    'LotSizeSquareFeet': 'LotSize',
}).copy()
df['CloseDate'] = pd.to_datetime(df['CloseDate'], errors='coerce')
for column in NUMERIC_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df.dropna(subset=['CloseDate', 'ClosePrice']).copy()
df['Month'] = df['CloseDate'].dt.to_period('M')
df = df.sort_values(['Month', 'CloseDate']).reset_index(drop=True)
df = df.loc[(df['PropertyType'] == 'Residential') & (df['PropertySubType'] == 'SingleFamilyResidence')].copy()
if df.empty:
    raise ValueError('The single-family residential filter returned no rows.')
print(f'Loaded {len(df):,} rows from {len(csv_files)} monthly file(s).')
print(f'Date range: {df["CloseDate"].min().date()} to {df["CloseDate"].max().date()}')

Loaded 794,263 rows from 30 monthly file(s).
Date range: 2022-01-01 to 2026-05-31


## Training-window experiment

The latest month remains untouched for the final test set. The preceding month is used as validation to choose `X` without using the final test month. Candidate windows are compared with a median-price baseline, and the lowest validation MAE is selected.

In [3]:
latest_month = df['Month'].max()
validation_month = latest_month - 1
window_candidates = [1, 2, 3, 4, 5, 6]
window_scores = []

for window in window_candidates:
    candidate_months = pd.period_range(validation_month - window, validation_month - 1, freq='M')
    if not set(candidate_months).issubset(set(df['Month'])):
        continue
    candidate_train = df.loc[df['Month'].isin(candidate_months), 'ClosePrice']
    validation_values = df.loc[df['Month'].eq(validation_month), 'ClosePrice']
    prediction = candidate_train.median()
    validation_mae = (validation_values - prediction).abs().mean()
    window_scores.append({'training_months': window, 'validation_mae': validation_mae})

window_scores = pd.DataFrame(window_scores).sort_values('validation_mae')
if window_scores.empty:
    raise ValueError('No candidate training window is fully covered by the available data.')
TRAINING_MONTHS = int(window_scores.iloc[0]['training_months'])
print(window_scores.to_string(index=False))
print(f'Selected TRAINING_MONTHS: {TRAINING_MONTHS}')

 training_months  validation_mae
               1   752138.402542
               6   752314.629263
               5   752844.240232
               2   752964.390924
               4   753147.942692
               3   754353.231604
Selected TRAINING_MONTHS: 1


In [4]:
test_month = latest_month
available_months = pd.period_range(df['Month'].min(), latest_month, freq='M')
training_months = pd.period_range(test_month - TRAINING_MONTHS, test_month - 1, freq='M')
if not set(training_months).issubset(set(available_months)):
    raise ValueError('The requested training window is not fully covered by the available data.')

train_mask = df['Month'].isin(training_months)
test_mask = df['Month'].eq(test_month)
train_raw = df.loc[train_mask].copy()
test_raw = df.loc[test_mask].copy()
print(f'Test month: {test_month}')
print(f'Training months: {training_months[0]} through {training_months[-1]} ({TRAINING_MONTHS} month(s))')
print(f'Raw training rows: {len(train_raw):,}; raw test rows: {len(test_raw):,}')

Test month: 2026-05
Training months: 2026-04 through 2026-04 (1 month(s))
Raw training rows: 23,412; raw test rows: 23,260


## Preprocessing decisions

- Rows missing `CloseDate` or the target `ClosePrice` are dropped because they cannot be assigned to a time split or evaluated.
- Missing numeric features are median-imputed using training rows only; companion `_missing` flags preserve the missingness signal.
- Missing categorical values are filled with `Unknown`, then one-hot encoded.
- Numeric feature columns are standardized using training medians and standard deviations only.
- The target `ClosePrice` is retained in its original units and is not normalized.

In [5]:
numeric_features = ['LivingArea', 'Bedrooms', 'Bathrooms', 'LotSize']
train_features = train_raw[FEATURE_COLUMNS].copy()
test_features = test_raw[FEATURE_COLUMNS].copy()

for column in numeric_features:
    train_features[f'{column}_missing'] = train_features[column].isna().astype('int8')
    test_features[f'{column}_missing'] = test_features[column].isna().astype('int8')
    median = train_features[column].median()
    if pd.isna(median):
        raise ValueError(f'Cannot impute {column}: training median is missing.')
    train_features[column] = train_features[column].fillna(median)
    test_features[column] = test_features[column].fillna(median)
    mean = train_features[column].mean()
    scale = train_features[column].std(ddof=0)
    if not np.isfinite(scale) or scale == 0:
        scale = 1.0
    train_features[column] = (train_features[column] - mean) / scale
    test_features[column] = (test_features[column] - mean) / scale

train_categories = train_features[CATEGORICAL_COLUMNS].fillna('Unknown').astype('string')
test_categories = test_features[CATEGORICAL_COLUMNS].fillna('Unknown').astype('string')
train_encoded = pd.get_dummies(train_categories, prefix=CATEGORICAL_COLUMNS, dtype='int8')
test_encoded = pd.get_dummies(test_categories, prefix=CATEGORICAL_COLUMNS, dtype='int8').reindex(columns=train_encoded.columns, fill_value=0)

scaled_numeric = numeric_features + [f'{column}_missing' for column in numeric_features]
train_processed = pd.concat([train_raw[['CloseDate', 'Month', 'ClosePrice']], train_features[scaled_numeric], train_encoded], axis=1)
test_processed = pd.concat([test_raw[['CloseDate', 'Month', 'ClosePrice']], test_features[scaled_numeric], test_encoded], axis=1)
train_processed['split'] = 'train'
test_processed['split'] = 'test'
processed_df = pd.concat([train_processed, test_processed], ignore_index=True)
processed_df['Month'] = processed_df['Month'].astype(str)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
processed_df.to_csv(OUTPUT_FILE, index=False)
print(f'Wrote {len(processed_df):,} cleaned rows to {OUTPUT_FILE}.')

Wrote 46,672 cleaned rows to outputs/02_preprocessed.csv.


In [6]:
assert set(processed_df.loc[processed_df['split'].eq('train'), 'Month']) == {str(month) for month in training_months}
assert set(processed_df.loc[processed_df['split'].eq('test'), 'Month']) == {str(test_month)}
assert processed_df['ClosePrice'].notna().all()
assert processed_df.drop(columns=['CloseDate', 'Month', 'ClosePrice', 'split']).notna().all().all()
print(processed_df.groupby('split')['Month'].agg(['min', 'max', 'count']))

           min      max  count
split                         
test   2026-05  2026-05  23260
train  2026-04  2026-04  23412


# Key Notes About Preprocessing

1. **Imports and settings:** Imports `Path`, NumPy, and pandas. It defines the input/output locations, the monthly file pattern, the candidate training-window setting, and the approved categorical, numeric, feature, and source columns.

2. **Load and organize the data:** Reads the 30 monthly CRMLS sold-property files using only the required columns. It renames the source bedroom, bathroom, and lot-size fields, converts dates and numeric values, drops rows that cannot be assigned to a time split or evaluated because `CloseDate` or `ClosePrice` is missing, keeps only residential single-family records, creates a monthly period, and sorts chronologically.

3. **Training-window experiment:** Keeps the latest month reserved for the final test set. It uses the immediately preceding month as validation and compares 1-to-6-month training windows using a median-price baseline and mean absolute error (MAE). The lowest validation MAE selects `TRAINING_MONTHS`; here, the selected window is 1 month.

4. **Create the time split:** Uses April 2026 as the training month and May 2026 as the test month. This is a chronological split rather than a random split, so the model is evaluated on the most recent available month.

5. **Preprocessing decisions:** Documents the missing-value policy. Numeric features use training medians for imputation and retain `_missing` indicator columns. Missing categorical values become `Unknown`. Numeric features are standardized from training statistics only, while the `ClosePrice` target remains in its original units.

6. **Fit and transform:** Applies those decisions to the training and test features. It creates one-hot encoded columns for `PropertyType` and `PropertySubType`, aligns test categories to the training columns, combines the processed features with the date, month, target, and split label, and writes `outputs/02_preprocessed.csv`.

7. **Final checks:** Confirms that the training and test months are exactly the intended months, that the target has no missing values, and that no processed feature cell is missing. The printed table reports the final row counts for each split.

## Final result

The cleaned CSV contains the selected training and test rows from residential single-family properties, with the latest month reserved for testing. The final processed features contain no missing values. The target is `ClosePrice`; the input features are the cleaned numeric fields, missingness indicators, and one-hot encoded categorical fields. No listing-only fields such as `ListPrice`, `OriginalListPrice`, `DaysOnMarket`, `PurchaseContractDate`, or `ListingContractDate` are included as inputs.